In [20]:
from langchain_core.tools import tool
@tool
def division(a:int , b:int) -> int:
    """Returns the division of a by b."""
    return a / b

In [21]:
print(division.name)
print(division.description)
print(division.args)

division
Returns the division of a by b.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [22]:
##async implementation
@tool
async def division(a:int , b:int) -> int:
    """Returns the division of a by b."""
    return a / b

In [23]:
from typing import Annotated,List

@tool
def multiply_by_max(
    a: Annotated[int, "A Value"],
    b: Annotated[List[int], "list of ints over which to take maximum"],
)->int:
    """Multiply a by the maximum of b"""

    return a* max(b)

In [24]:
print(multiply_by_max.args)

{'a': {'description': 'A Value', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'list of ints over which to take maximum', 'items': {'type': 'integer'}, 'title': 'B', 'type': 'array'}}


## structures tool

the StructureTool.from_function class method a bot more configurableity than tool@decorator

In [25]:
from langchain_core.tools import StructuredTool
def multiply(a:int, b:int)->int:
    """ multiply two numbers """
    return a*b

async def amultiply(a:int, b:int)->int:
    """ multiply two numbers """
    return a*b

calculator = StructuredTool.from_function(func=multiply,coroutine=amultiply)

print(calculator.invoke({"a":2 ,"b":3}))
print(await calculator.ainvoke({"a":5 ,"b":3}))


6
15


## Inbuilt Tools

In [26]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper=WikipediaAPIWrapper(top_k_results=5,doc_content_chars_max=500)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(tool.invoke({"query":"langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Vector database
Summary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in v


In [27]:
from langchain_community.utilities import ArXivAPIWrapper
tool = ArXivAPIWrapper()

tool.run("1706.03762")

ImportError: cannot import name 'ArXivAPIWrapper' from 'langchain_community.utilities' (c:\Users\sajal\OneDrive\Desktop\gen-ai\.venv\Lib\site-packages\langchain_community\utilities\__init__.py)

In [28]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [29]:
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [30]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [ ]:
tool.invoke("what is recent war news about America and Iran?")

{'query': 'what is recent war news about America and Iran?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.nytimes.com/live/2026/04/08/world/iran-war-trump-news',
   'title': 'Iran War Live Updates: Fragile Cease-Fire Takes Hold as Both Sides Claim ...',
   'content': "A timeline: The cease-fire was announced on the 39th day of the U.S.-Israeli war with Iran. ... War decimated Iran's Leadership and pushed up a",
   'score': 0.99351174,
   'raw_content': None},
  {'url': 'https://www.economist.com/middle-east-and-africa/2026/04/08/iran-and-america-agree-to-pause-their-war',
   'title': 'Iran and America agree to pause their war',
   'content': 'Iran and America agree to pause their war · A two-week truce is a start but questions remain about a more enduring deal · More from Middle East &',
   'score': 0.9911527,
   'raw_content': None},
  {'url': 'https://www.youtube.com/watch?v=2l8Cc7qXWqw',
   'title': 'Trump Announces Ceasefire LIVE |

## Call Tools with LLM

In [31]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper=WikipediaAPIWrapper(top_k_results=5,doc_content_chars_max=500)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(tool.invoke({"query":"langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Vector database
Summary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in v


In [32]:
from langchain_core.tools import tool
@tool
def add(a:int,b:int)->int:
    """Returns the sum of a and b."""
    return a+b

@tool
def multiply(a:int,b:int)->int:
    """Returns the product of a and b."""
    return a*b

In [33]:
tools=[wiki_tool,add,multiply,tavily_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\sajal\\OneDrive\\Desktop\\gen-ai\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=5, lang='en', load_all_available_meta=False, doc_content_chars_max=500)),
 StructuredTool(name='add', description='Returns the sum of a and b.', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x0000018BF8EDF420>),
 StructuredTool(name='multiply', description='Returns the product of a and b.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x0000018BF8EDEFC0>),
 TavilySearch(max_results=5, topic='general', api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))]

In [34]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("qwen/qwen3-32b", model_provider="groq")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018BF8F6CCD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018BF8F6D6D0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [35]:
llm_with_tools=llm.bind_tools(tools)
llm_with_tools

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018BF8F6CCD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018BF8F6D6D0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'wikipedia', 'description': 'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.', 'parameters': {'properties': {'query': {'description': 'query to look up on wikipedia', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type': 'function

In [40]:
from langchain_core.messages import HumanMessage

In [44]:

querey="what is 2*4"
messages = [HumanMessage(querey)]
response=llm_with_tools.invoke(querey)
print(response)

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking "what is 2*4". Let me think about how to approach this.\n\nFirst, I need to recognize that 2 multiplied by 4 is a basic arithmetic problem. The user is likely looking for the product of these two numbers. \n\nLooking at the available tools, there\'s a "multiply" function that takes two integers as parameters. The description says it returns the product of a and b. That\'s exactly what I need here.\n\nSo, I should call the multiply function with a=2 and b=4. There\'s no need to use any other tool like Wikipedia or the search engine because this is a straightforward math problem. \n\nI need to make sure the parameters are correctly formatted as integers. The function requires "a" and "b" as required parameters, both of type integer. \n\nTherefore, the correct tool call is to use the multiply function with the arguments {"a": 2, "b": 4}.\n', 'tool_calls': [{'id': 'zzz9xh3xr', 'function': {'arguments': '{"a":2,"b"

In [43]:
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 4},
  'id': 'ae04tnva7',
  'type': 'tool_call'}]

In [45]:
for tool_call in response.tool_calls:
    selected_tool = {"add": add, "multiply": multiply,"wikipedia":wiki_tool,"tavily_search":tavily_tool}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    
messages


[HumanMessage(content='what is 2*4', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='8', name='multiply', tool_call_id='zzz9xh3xr')]

In [46]:
llm_with_tools.invoke(messages)

AIMessage(content='The product of 2 and 4 is 8.\n\n', additional_kwargs={'reasoning_content': 'Okay, the user asked "what is 2*4". Let me think. The tools available include the multiply function, which takes two integers and returns their product. Since 2 and 4 are both integers, I can use the multiply function here. I don\'t need to use any other tools like Wikipedia or the search engine because this is a straightforward arithmetic problem. So, calling the multiply function with a=2 and b=4 should give the answer. Let me check the parameters again to make sure. The function requires "a" and "b" as integers. Yep, that\'s correct. So the tool call would be {"name": "multiply", "arguments": {"a": 2, "b": 4}}. The response should be 8.\n', 'tool_calls': [{'id': 'kh1vmk2v2', 'function': {'arguments': '{"a":2,"b":4}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 199, 'prompt_tokens': 1951, 'total_tokens': 2150, 'completion_time': 0.3489

In [47]:
querey1 = "what is capital of india and what is 12*4"
messages = [HumanMessage(querey1)]
ai_msg=llm_with_tools.invoke(messages)
ai_msg


AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking two separate questions here. First, they want to know the capital of India. I remember that the capital of India is New Delhi. But maybe I should verify that using the Wikipedia function to be sure. I can call the wikipedia function with the query "capital of India" to get accurate information.\n\nSecond part of the question is a math problem: 12 multiplied by 4. That\'s straightforward. The multiply function can handle that. So I need to call the multiply function with a=12 and b=4. \n\nWait, do I need to use the functions for both parts? For the capital of India, using Wikipedia makes sense. For the math problem, using the multiply function is the right approach. I should structure the tool calls accordingly. Let me check if there are any other parameters needed for the functions. The Wikipedia function just needs the query, and the multiply function needs the two integers. \n\nSo, first tool call